# **Production Fraud Detection Platform**
## **Notebook 1: Data Ingestion & Exploratory Data Analysis (EDA)**

---

###  Project Overview
This notebook is **Part 1** of a full end-to-end Production Fraud Detection 
Platform built on AWS. We build a real-time system capable of scoring 
1,000+ transactions per second with sub-100ms latency.

### Business Problem
Financial fraud costs the global economy **$485 billion annually**.
Traditional rule-based systems catch only 30-40% of fraud cases and 
generate too many false positives, blocking legitimate customers.

**Our goal:** Build a Machine Learning system that:
- Detects fraud with **AUC-ROC > 0.96**
- Scores transactions in **< 100ms** in real-time
- Reduces false positives by **40%** vs rule-based systems
- Automatically retrains when model performance degrades

---

### 📊 Dataset: IEEE-CIS Fraud Detection
| Property | Details |
|----------|---------|
| **Source** | IEEE Computational Intelligence Society + Vesta Corporation |
| **Transactions** | 590,540 real e-commerce transactions |
| **Features** | 431 features (transaction + identity) |
| **Fraud Rate** | ~3.5% (highly imbalanced) |
| **Time Period** | Real production data from Vesta's fraud protection system |
| **Files** | train_transaction.csv + train_identity.csv |

---

### 🗺️ Notebook Structure
| Step | Description |
|------|-------------|
| **Step 1** | Environment Setup & Library Import |
| **Step 2** | Data Loading from S3 |
| **Step 3** | First Look at the Data |
| **Step 4** | Target Variable Analysis (Fraud Distribution) |
| **Step 5** | Transaction Features Analysis |
| **Step 6** | Identity Features Analysis |
| **Step 7** | Correlation Analysis |
| **Step 8** | Key Insights & Next Steps |

---
> 👨‍💻 **Author:** Armand Junior Dongmo Notue  
> 📅 **Date:** March 2026  
> ☁️ **Platform:** AWS SageMaker Studio  
> 🔗 **GitHub:** github.com/armand/fraud-detection-platform

## **Environment Setup & Library Import**

### Introduction
Before loading our data, we set up our entire Python environment.
We import all libraries needed for:
- **Data manipulation:** Pandas, NumPy
- **Visualization:** Matplotlib, Seaborn, Plotly
- **Cloud storage:** Boto3 (AWS SDK)
- **ML later:** Scikit-learn, Imbalanced-learn

> This is a **production notebook**; every library choice has a reason.
> We use Plotly for interactive charts (better for presentations & GitHub).

In [1]:
# Environment Setup & Library Import


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import boto3
import warnings
import os
import gc
from datetime import datetime

# Settings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: '%.3f' % x)
plt.style.use('seaborn-v0_8-darkgrid')

# ── Project Configuration ──────────────────────────────────
CONFIG = {
    "bucket"        : "fraud-detection-mlproject-armandjunior",
    "raw_prefix"    : "raw-data",
    "processed_prefix": "processed-data",
    "local_path"    : "/home/sagemaker-user/fraud-detection-platform/data/raw",
    "author"        : "Armand Junior Dongmo Notue",
    "project"       : "Production Fraud Detection Platform",
    "date"          : datetime.now().strftime("%Y-%m-%d")
}

# ── Print Summary ──────────────────────────────────────────
print("=" * 55)
print(f"  {CONFIG['project']}")
print("=" * 55)
print(f"  Author  : {CONFIG['author']}")
print(f"  Date    : {CONFIG['date']}")
print(f"   Bucket  : {CONFIG['bucket']}")
print(f"  Raw data: {CONFIG['local_path']}")
print("=" * 55)
print(f"\n✅ Pandas    : {pd.__version__}")
print(f"✅ NumPy     : {np.__version__}")
print(f"✅ Boto3     : {boto3.__version__}")
print(f"✅ Matplotlib: {plt.matplotlib.__version__}")
print("\n Environment ready, Let's detect some fraud!")


  Production Fraud Detection Platform
  Author  : Armand Junior Dongmo Notue
  Date    : 2026-03-10
   Bucket  : fraud-detection-mlproject-armandjunior
  Raw data: /home/sagemaker-user/fraud-detection-platform/data/raw

✅ Pandas    : 2.3.3
✅ NumPy     : 1.26.4
✅ Boto3     : 1.37.3
✅ Matplotlib: 3.10.8

 Environment ready, Let's detect some fraud!


## **Data Loading from S3**

### Introduction
We load our two IEEE-CIS files directly from our **AWS S3 data lake**.

The IEEE-CIS dataset has two separate files that must be **joined**:
- **train_transaction.csv**; 590,540 rows, 394 columns (core transaction data)
- **train_identity.csv**; 144,233 rows, 41 columns (device & identity data)

Not all transactions have identity information, only ~25% do.
This is realistic: many online purchases are made as guests.

We use **memory optimization techniques** to load this data efficiently
because at 1.4GB combined, naive loading would crash a small instance.

In [2]:
# Data Loading - OPTIMIZED VERSION

import pandas as pd
import numpy as np
import gc

print(" Loading IEEE-CIS Dataset (Optimized)...")
print("-" * 50)

TRANSACTION_COLS = [
    'TransactionID', 'isFraud', 'TransactionDT',
    'TransactionAmt', 'ProductCD', 'card1', 'card2',
    'card3', 'card4', 'card5', 'card6', 'addr1', 'addr2',
    'dist1', 'dist2', 'P_emaildomain', 'R_emaildomain',
    'C1','C2','C3','C4','C5','C6','C7','C8','C9','C10',
    'C11','C12','C13','C14',
    'D1','D2','D3','D4','D5','D6','D7','D8','D9','D10',
    'D11','D12','D13','D14','D15',
    'M1','M2','M3','M4','M5','M6','M7','M8','M9',
    'V1','V2','V3','V4','V5','V6','V7','V8','V9','V10'
]

IDENTITY_COLS = [
    'TransactionID', 'DeviceType', 'DeviceInfo',
    'id_01','id_02','id_03','id_04','id_05','id_06',
    'id_07','id_08','id_09','id_10','id_11','id_12',
    'id_13','id_14','id_15','id_16','id_17','id_18',
    'id_19','id_20','id_30','id_31','id_32','id_33','id_34'
]

# ── Load Transaction Data ──────────────────────────────────
print("\n Loading train_transaction.csv...")
train_transaction = pd.read_csv(
    "/home/sagemaker-user/fraud-detection-platform/data/raw/train_transaction.csv",
    usecols=TRANSACTION_COLS,
    dtype={'isFraud': np.int8,
           'TransactionDT': np.int32,
           'TransactionAmt': np.float32}
)
print(f"   Shape: {train_transaction.shape}")
print(f"   Memory: {train_transaction.memory_usage().sum()/1024**2:.1f} MB")

# ── Load Identity Data ─────────────────────────────────────
print("\n🪪  Loading train_identity.csv...")
train_identity = pd.read_csv(
    "/home/sagemaker-user/fraud-detection-platform/data/raw/train_identity.csv",
    usecols=IDENTITY_COLS
)
print(f"   Shape: {train_identity.shape}")
print(f"   Memory: {train_identity.memory_usage().sum()/1024**2:.1f} MB")

# ── Merge ──────────────────────────────────────────────────
print("\n🔗 Merging transaction + identity data...")
df = train_transaction.merge(
    train_identity,
    on='TransactionID',
    how='left'
)

del train_transaction, train_identity
gc.collect()

print("\n" + "=" * 50)
print("✅ Data loaded successfully!")
print(f"   Transactions : {df.shape[0]:,}")
print(f"   Features     : {df.shape[1]:,}")
print(f"   Memory       : {df.memory_usage().sum()/1024**2:.1f} MB")
print(f"   Fraud cases  : {df['isFraud'].sum():,}")
print(f"   Normal cases : {(df['isFraud']==0).sum():,}")
print(f"   Fraud rate   : {df['isFraud'].mean()*100:.2f}%")
print("=" * 50)

 Loading IEEE-CIS Dataset (Optimized)...
--------------------------------------------------

 Loading train_transaction.csv...
   Shape: (590540, 65)
   Memory: 284.4 MB

🪪  Loading train_identity.csv...
   Shape: (144233, 28)
   Memory: 30.8 MB

🔗 Merging transaction + identity data...

✅ Data loaded successfully!
   Transactions : 590,540
   Features     : 92
   Memory       : 406.1 MB
   Fraud cases  : 20,663
   Normal cases : 569,877
   Fraud rate   : 3.50%


###  What we accomplished:
We successfully loaded and merged **two real-world datasets** from 
Vesta Corporation's fraud protection system into a single DataFrame 
ready for analysis.

###  Key Numbers to Remember:
| Metric | Value |
|--------|-------|
| **Total transactions** | 590,540 |
| **Total features** | 92 |
| **Memory used** | 406 MB |
| **Fraud cases** | 20,663 |
| **Normal cases** | 569,877 |
| **Fraud rate** | 3.50% |

###  Key Takeaways:

**1. Class Imbalance is our #1 challenge** 
With only **3.50% fraud rate**, our dataset is highly imbalanced.
For every 1 fraud transaction, there are **27 legitimate ones**.
A model that predicts "NOT FRAUD" for everything would score
**96.5% accuracy** but catch **zero fraud**; completely useless!

We will address this later using:
- `scale_pos_weight` parameter in XGBoost
- SMOTE oversampling technique
- Evaluation using **AUC-PR** and **F1-Score**, not accuracy

**2. Smart memory optimization works** 
By selecting only 92 of 431 columns and specifying dtypes upfront,
we reduced memory from ~4GB to **406MB**, a **90% reduction**.
This allows us to run on a free-tier instance (`ml.t3.medium`)
instead of paying for a large GPU instance.

**3. Identity data is sparse** 
Only **144,233 out of 590,540** transactions have identity data (~24%).
This means ~76% of transactions were made without device/identity info typical of guest checkout e-commerce purchases.
We will handle these missing values carefully in feature engineering.

###  Next Step:
Now that data is loaded, we do a **deep first look** at the structure,
data types, missing values, and basic statistics of our dataset.

## **First Look at the Data**

Before any analysis, we always do a "first look" at our data.
This tells us:
- **What does the data look like?** (sample rows)
- **What are the data types?** (numeric vs categorical)
- **How much data is missing?** (null values)
- **What are the basic statistics?** (min, max, mean)

This step is what separates professional data scientists
from beginners — you NEVER skip exploratory analysis!

In [ ]:
# First Look at the Data


print("=" * 55)
print("   FIRST LOOK AT THE DATA")
print("=" * 55)

# ── Basic Info ─────────────────────────────────────────────
print(f"\n Dataset Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\n Column names and types:")
print("-" * 40)

# Show column types summary
type_counts = df.dtypes.value_counts()
for dtype, count in type_counts.items():
    print(f"   {str(dtype):<12} → {count:>3} columns")

# ── Sample rows ────────────────────────────────────────────
print(f"\n First 3 rows of key columns:")
key_cols = ['TransactionID','isFraud','TransactionAmt',
            'ProductCD','card4','card6',
            'P_emaildomain','DeviceType']
print(df[key_cols].head(3).to_string(index=False))

# ── Missing Values Analysis ────────────────────────────────
print(f"\n Missing Values Analysis:")
print("-" * 40)
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
}).query('`Missing Count` > 0').sort_values('Missing %', ascending=False)

print(f"   Columns with missing values: {len(missing_df)} out of {df.shape[1]}")
print(f"   Top 10 columns with most missing data:")
print(missing_df.head(10).to_string())

# ── Basic Statistics ───────────────────────────────────────
print(f"\n📈 Transaction Amount Statistics:")
print("-" * 40)
amt_stats = df['TransactionAmt'].describe()
print(f"   Min    : ${amt_stats['min']:.2f}")
print(f"   Mean   : ${amt_stats['mean']:.2f}")
print(f"   Median : ${df['TransactionAmt'].median():.2f}")
print(f"   Max    : ${amt_stats['max']:.2f}")
print(f"   Std    : ${amt_stats['std']:.2f}")

print("\n First look complete!")

###  What we discovered:

| Finding | Detail | Action |
|---------|--------|--------|
| **92 features** | Mix of numeric (67) + categorical (23) | Need encoding |
| **72 columns with NaN** | 78% of features have missing data | Smart imputation |
| **4 columns >90% missing** | id_07, id_08, dist2, D7 | Will DROP |
| **Transaction range** | $0.25 to $31,937 | Need log transform |
| **Right-skewed amounts** | Mean $135 vs Median $69 | Confirms skewness |

### Key Takeaways:

**1. Missing data is our biggest feature engineering challenge**
72 out of 92 columns (78%) have missing values.
Some columns like `id_07` and `id_08` are 99.1% empty, we will DROP them.
But missingness itself can be a fraud signal a transaction with
NO device info is more likely fraudulent than one with full identity.
We will create **binary missingness flags** as new features.

**2. Transaction amounts are highly skewed**
The median ($68.77) is half the mean ($135.03), classic right skew.
We will apply **log transformation**: `log(TransactionAmt + 1)`
to normalize this distribution for our models.

**3. Mixed data types require a dual pipeline**
- Numerical features → StandardScaler + median imputation
- Categorical features → LabelEncoder + mode imputation

### ➡️ Next Step:
We analyze our **target variable (fraud distribution)** in depth
using visualizations to understand fraud patterns across time,
amount, product type, and card type.

## **Target Variable Analysis — Fraud Distribution**

The target variable `isFraud` is the heart of our entire project.
Before building any model, we must deeply understand:
- **How much fraud exists?** (overall distribution)
- **When does fraud happen?** (time patterns)
- **How much do fraudsters spend?** (amount patterns)
- **What products are targeted?** (product patterns)
- **Which cards are used?** (card type patterns)

These insights directly drive our **feature engineering decisions**
and help us explain our model to business stakeholders.

In [ ]:

# Target Variable Analysis — Fraud Distribution

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

print("=" * 55)
print("  TARGET VARIABLE ANALYSIS")
print("=" * 55)

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle(' Fraud Distribution Analysis\nIEEE-CIS Fraud Detection Dataset',
             fontsize=16, fontweight='bold', y=1.02)

# ── Plot 1: Overall Fraud Distribution ─────────────────────
ax1 = axes[0, 0]
fraud_counts = df['isFraud'].value_counts()
colors = ['#2ecc71', '#e74c3c']
bars = ax1.bar(['Legitimate', 'Fraud'],
               fraud_counts.values,
               color=colors, width=0.5, edgecolor='black')
ax1.set_title('Overall Transaction Distribution', fontweight='bold')
ax1.set_ylabel('Number of Transactions')
for bar, count in zip(bars, fraud_counts.values):
    ax1.text(bar.get_x() + bar.get_width()/2.,
             bar.get_height() + 2000,
             f'{count:,}\n({count/len(df)*100:.1f}%)',
             ha='center', va='bottom', fontweight='bold')
ax1.set_ylim(0, fraud_counts.max() * 1.15)

# ── Plot 2: Transaction Amount — Fraud vs Legitimate ───────
ax2 = axes[0, 1]
legitimate_amt = df[df['isFraud']==0]['TransactionAmt']
fraud_amt = df[df['isFraud']==1]['TransactionAmt']
ax2.hist(np.log1p(legitimate_amt), bins=50, alpha=0.6,
         color='#2ecc71', label=f'Legitimate (n={len(legitimate_amt):,})')
ax2.hist(np.log1p(fraud_amt), bins=50, alpha=0.6,
         color='#e74c3c', label=f'Fraud (n={len(fraud_amt):,})')
ax2.set_title('Transaction Amount Distribution\n(Log Scale)', fontweight='bold')
ax2.set_xlabel('Log(TransactionAmt + 1)')
ax2.set_ylabel('Frequency')
ax2.legend()

# ── Plot 3: Fraud by Product Type ──────────────────────────
ax3 = axes[0, 2]
product_fraud = df.groupby('ProductCD')['isFraud'].agg(['sum','count'])
product_fraud['rate'] = product_fraud['sum'] / product_fraud['count'] * 100
product_fraud = product_fraud.sort_values('rate', ascending=True)
colors_prod = ['#e74c3c' if r > 5 else '#3498db'
               for r in product_fraud['rate']]
bars3 = ax3.barh(product_fraud.index,
                 product_fraud['rate'],
                 color=colors_prod, edgecolor='black')
ax3.set_title('Fraud Rate by Product Type', fontweight='bold')
ax3.set_xlabel('Fraud Rate (%)')
for bar, val in zip(bars3, product_fraud['rate']):
    ax3.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
             f'{val:.1f}%', va='center', fontweight='bold')

# ── Plot 4: Fraud by Card Type ─────────────────────────────
ax4 = axes[1, 0]
card_fraud = df.groupby('card4')['isFraud'].agg(['sum','count'])
card_fraud['rate'] = card_fraud['sum'] / card_fraud['count'] * 100
card_fraud = card_fraud.dropna().sort_values('rate', ascending=False)
bars4 = ax4.bar(card_fraud.index,
                card_fraud['rate'],
                color='#9b59b6', edgecolor='black')
ax4.set_title('Fraud Rate by Card Network', fontweight='bold')
ax4.set_ylabel('Fraud Rate (%)')
ax4.set_xlabel('Card Network')
for bar, val in zip(bars4, card_fraud['rate']):
    ax4.text(bar.get_x() + bar.get_width()/2.,
             bar.get_height() + 0.05,
             f'{val:.1f}%', ha='center', fontweight='bold')

# ── Plot 5: Fraud by Card Category ─────────────────────────
ax5 = axes[1, 1]
card6_fraud = df.groupby('card6')['isFraud'].agg(['sum','count'])
card6_fraud['rate'] = card6_fraud['sum'] / card6_fraud['count'] * 100
card6_fraud = card6_fraud.dropna().sort_values('rate', ascending=False)
bars5 = ax5.bar(card6_fraud.index,
                card6_fraud['rate'],
                color='#e67e22', edgecolor='black')
ax5.set_title('Fraud Rate by Card Category', fontweight='bold')
ax5.set_ylabel('Fraud Rate (%)')
ax5.set_xlabel('Card Category')
for bar, val in zip(bars5, card6_fraud['rate']):
    ax5.text(bar.get_x() + bar.get_width()/2.,
             bar.get_height() + 0.05,
             f'{val:.1f}%', ha='center', fontweight='bold')

# ── Plot 6: Transaction Amount — Box Plot ──────────────────
ax6 = axes[1, 2]
data_box = [np.log1p(legitimate_amt.sample(5000, random_state=42)),
            np.log1p(fraud_amt)]
bp = ax6.boxplot(data_box,
                 labels=['Legitimate', 'Fraud'],
                 patch_artist=True,
                 notch=True)
bp['boxes'][0].set_facecolor('#2ecc71')
bp['boxes'][1].set_facecolor('#e74c3c')
ax6.set_title('Amount Distribution Comparison\n(Log Scale)', fontweight='bold')
ax6.set_ylabel('Log(TransactionAmt + 1)')

plt.tight_layout()
plt.savefig('fraud_distribution_analysis.png',
            dpi=150, bbox_inches='tight')
plt.show()

# ── Print Key Statistics ────────────────────────────────────
print("\n KEY FRAUD STATISTICS:")
print("-" * 40)
print(f"   Overall fraud rate    : {df['isFraud'].mean()*100:.2f}%")
print(f"   Avg fraud amount      : ${fraud_amt.mean():.2f}")
print(f"   Avg legitimate amount : ${legitimate_amt.mean():.2f}")
print(f"   Median fraud amount   : ${fraud_amt.median():.2f}")
print(f"   Median legit amount   : ${legitimate_amt.median():.2f}")
print(f"\n Fraud by Product Type:")
for prod, row in product_fraud.iterrows():
    print(f"   {prod}: {row['rate']:.1f}% fraud rate "
          f"({int(row['sum']):,} fraud / {int(row['count']):,} total)")
print(f"\n Fraud by Card Network:")
for card, row in card_fraud.iterrows():
    print(f"   {card}: {row['rate']:.1f}% fraud rate")
print("\n Fraud distribution analysis complete!")

### 6 Key Discoveries from our Charts:

| Finding | Detail | Model Impact |
|---------|--------|-------------|
| **3.5% fraud rate** | Severe class imbalance | Use weighted loss |
| **Product C = 11.7%** | 6x higher than Product W | Top feature |
| **Discover = 7.7%** | 2.7x higher than Amex | Strong signal |
| **Credit = 6.7%** | 2.8x higher than debit | Strong signal |
| **Fraud avg $149** | Slightly > legit $134 | Weak signal alone |
| **Similar distributions** | Fraudsters mimic normal | Need complex model |

### Key Takeaways:

**1. Product type is our strongest signal**
Product C has an 11.7% fraud rate — nearly 6x higher than Product W.
This single feature will appear in our top 5 most important features.

**2. Fraudsters are smart**
Transaction amounts between fraud and legitimate are very similar.
Simple threshold rules ("block transactions > $500") would miss
most fraud. This confirms we need Machine Learning, not rules.

**3. Card type matters significantly**
Credit cards (6.7%) vs debit cards (2.4%) — a 2.8x difference.
Discover network (7.7%) vs Amex (2.9%) — a 2.7x difference.
Both `card4` and `card6` will be important model features.

**4. Class imbalance demands special treatment**
With 96.5% legitimate vs 3.5% fraud, we will use:
- `scale_pos_weight = 569877/20663 ≈ 27.6` in XGBoost
- Evaluation on **AUC-PR** and **F1-Score**, not accuracy
- SMOTE oversampling for minority class augmentation

### ➡️ Next Step:
We analyze **missing values in depth** and build our complete
**feature engineering pipeline** — transforming raw data into
powerful predictive signals for our XGBoost model.

## **Missing Values Deep Dive**

We discovered 72 out of 92 columns have missing values.
Now we go deeper to make smart decisions:
- **DROP** columns with >90% missing (no predictive value)
- **KEEP + FLAG** columns where missingness = fraud signal
- **IMPUTE** columns with <50% missing

This step directly determines the quality of our final model.
A bad imputation strategy can destroy model performance!

### 🎯 Our Strategy:
| Missing % | Action | Reason |
|-----------|--------|--------|
| > 90% | DROP column | Almost no information |
| 50–90% | FLAG + impute | Missingness may signal fraud |
| < 50% | Impute only | Enough data to fill safely |

In [ ]:
# Missing Values Deep Dive


print("=" * 55)
print("  MISSING VALUES DEEP DIVE")
print("=" * 55)

# ── Step 1: Full Missing Values Table ──────────────────────
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct,
    'Remaining': len(df) - missing,
}).query('`Missing Count` > 0').sort_values('Missing %', ascending=False)

# ── Step 2: Categorize columns ─────────────────────────────
drop_cols = missing_df[missing_df['Missing %'] > 90].index.tolist()
flag_cols = missing_df[(missing_df['Missing %'] > 50) & 
                       (missing_df['Missing %'] <= 90)].index.tolist()
impute_cols = missing_df[missing_df['Missing %'] <= 50].index.tolist()

print(f"\n Missing Values Summary:")
print(f"   Total columns        : {df.shape[1]}")
print(f"   Columns with no NaN  : {df.shape[1] - len(missing_df)}")
print(f"   Columns with NaN     : {len(missing_df)}")
print(f"\n     DROP  (>90% missing) : {len(drop_cols)} columns")
print(f"     FLAG  (50-90% missing): {len(flag_cols)} columns")
print(f"     IMPUTE (<50% missing) : {len(impute_cols)} columns")

# ── Step 3: Visualize Missing Values ───────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('Missing Values Analysis', fontsize=14, fontweight='bold')

# Plot 1: Missing % by column
ax1 = axes[0]
colors_miss = ['#e74c3c' if p > 90 else '#f39c12' if p > 50 
               else '#3498db' for p in missing_df['Missing %']]
bars = ax1.barh(range(len(missing_df)), 
                missing_df['Missing %'],
                color=colors_miss)
ax1.set_yticks(range(len(missing_df)))
ax1.set_yticklabels(missing_df.index, fontsize=8)
ax1.axvline(x=90, color='red', linestyle='--', 
            linewidth=2, label='90% threshold (DROP)')
ax1.axvline(x=50, color='orange', linestyle='--', 
            linewidth=2, label='50% threshold (FLAG)')
ax1.set_xlabel('Missing %')
ax1.set_title('Missing % per Column', fontweight='bold')
ax1.legend()

# Plot 2: Strategy pie chart
ax2 = axes[1]
strategy_counts = [len(drop_cols), len(flag_cols), 
                   len(impute_cols), df.shape[1]-len(missing_df)]
strategy_labels = [f'DROP\n(>90%)\n{len(drop_cols)} cols',
                   f'FLAG+Impute\n(50-90%)\n{len(flag_cols)} cols',
                   f'Impute only\n(<50%)\n{len(impute_cols)} cols',
                   f'Complete\n(no NaN)\n{df.shape[1]-len(missing_df)} cols']
colors_pie = ['#e74c3c', '#f39c12', '#3498db', '#2ecc71']
ax2.pie(strategy_counts, labels=strategy_labels,
        colors=colors_pie, autopct='%1.1f%%',
        startangle=90, textprops={'fontsize': 9})
ax2.set_title('Column Treatment Strategy', fontweight='bold')

plt.tight_layout()
plt.savefig('missing_values_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Step 4: Check if missingness correlates with fraud ─────
print(f"\n Does missingness correlate with fraud?")
print("-" * 45)
for col in flag_cols[:5]:  # Check top 5 flag columns
    missing_mask = df[col].isnull()
    fraud_rate_missing = df[missing_mask]['isFraud'].mean() * 100
    fraud_rate_present = df[~missing_mask]['isFraud'].mean() * 100
    print(f"   {col}:")
    print(f"      When MISSING → fraud rate: {fraud_rate_missing:.1f}%")
    print(f"      When PRESENT → fraud rate: {fraud_rate_present:.1f}%")
    diff = fraud_rate_missing - fraud_rate_present
    signal = " STRONG SIGNAL!" if abs(diff) > 2 else "→ weak signal"
    print(f"      Difference: {diff:+.1f}% {signal}")

# ── Step 5: Apply Strategy ─────────────────────────────────
print(f"\n Applying Missing Value Strategy...")
print("-" * 45)

# DROP high-missing columns
df_clean = df.drop(columns=drop_cols)
print(f"    Dropped {len(drop_cols)} columns (>90% missing)")

# CREATE missingness flags for flag_cols
flags_created = 0
for col in flag_cols:
    if col in df_clean.columns:
        df_clean[f'{col}_missing'] = df_clean[col].isnull().astype(np.int8)
        flags_created += 1
print(f"    Created {flags_created} missingness flag features")

# IMPUTE numerical columns with median
num_imputed = 0
cat_imputed = 0
for col in df_clean.columns:
    if df_clean[col].isnull().sum() > 0:
        if df_clean[col].dtype in ['float64', 'float32', 
                                    'int64', 'int32', 'int8']:
            df_clean[col] = df_clean[col].fillna(df_clean[col].median())
            num_imputed += 1
        else:
            df_clean[col] = df_clean[col].fillna('Unknown')
            cat_imputed += 1

print(f"    Imputed {num_imputed} numerical columns (median)")
print(f"    Imputed {cat_imputed} categorical columns (Unknown)")

print(f"\n{'='*55}")
print(f" MISSING VALUES HANDLED SUCCESSFULLY!")
print(f"   Original shape  : {df.shape}")
print(f"   Clean shape     : {df_clean.shape}")
print(f"   Remaining NaN   : {df_clean.isnull().sum().sum()}")
print(f"   New flag features: {flags_created}")
print(f"{'='*55}")

###  What we accomplished:
Transformed a dataset with 72 missing-value columns into a
**clean 124-feature dataset with zero NaN values**.

###  Treatment Summary:
| Strategy | Columns | Action |
|----------|---------|--------|
| **DROP** | 5 cols (>90% missing) | Removed — no value |
| **FLAG + Impute** | 37 cols (50-90%) | Created binary flag + imputed |
| **Impute only** | 30 cols (<50%) | Median/Unknown fill |
| **Complete** | 20 cols (no NaN) | No action needed |

###  Key Takeaways:

**1. Missingness is a powerful fraud signal**
All 5 top FLAG columns showed STRONG SIGNAL (>8% difference):
- When D12 is PRESENT → 11.7% fraud rate
- When D12 is MISSING → only 2.5% fraud rate
- This 9.3% difference is one of our strongest predictors!

**2. We engineered 37 new features for free**
By creating binary missingness flags, we added 37 new features
that capture fraud patterns invisible to simple imputation.
This is advanced feature engineering that separates
professional data scientists from beginners.

**3. Dataset grew from 92 → 124 features**
More high-quality features = better model performance.
Our target of AUC-ROC > 0.96 is now more achievable!

**4. Zero remaining NaN values**
Our dataset is now fully clean and ready for modeling.
Numerical columns filled with median (robust to outliers).
Categorical columns filled with 'Unknown' (preserves category).

### ➡️ Next Step:
Now we build **advanced engineered features** — ratio features,
time features, aggregation features, and encoding categoricals.
This is where we create the signals that push AUC from 0.85 → 0.96!

## **Feature Engineering**

Now running on `ml.t3.large` (4GB RAM) — we use the 
**complete feature engineering pipeline** with all columns.

We create 5 categories of powerful features:
| Category | Count | Examples |
|----------|-------|---------|
| **Amount features** | 4 | log, sqrt, decimal, isround |
| **Time features** | 5 | hour, day, week, is_night, is_weekend |
| **Card aggregations** | 8 | count, mean, std, ratio per card |
| **Email features** | 3 | match, missing, high_risk |
| **Missingness flags** | 37 | binary signal for each NaN column |

> 💡 All results are checkpointed to S3 after processing.
> If the kernel crashes, we reload from S3 — not from scratch!

In [ ]:

#  Feature Engineering 


import pandas as pd
import numpy as np
import gc
import boto3
from sklearn.preprocessing import LabelEncoder

print("=" * 55)
print("   FEATURE ENGINEERING — FULL PIPELINE")
print("=" * 55)

# ── CONFIG ──────────────────────────────────────────────────
LOCAL_PATH = "/home/sagemaker-user/fraud-detection-platform/data/raw"
BUCKET     = "fraud-detection-mlproject-armand"

# ── STEP 1: RELOAD DATA ─────────────────────────────────────
print("\n Step 1: Loading raw data...")

TRANSACTION_COLS = [
    'TransactionID', 'isFraud', 'TransactionDT',
    'TransactionAmt', 'ProductCD',
    'card1', 'card2', 'card3', 'card4', 'card5', 'card6',
    'addr1', 'addr2', 'dist1', 'dist2',
    'P_emaildomain', 'R_emaildomain',
    'C1','C2','C3','C4','C5','C6','C7','C8','C9','C10',
    'C11','C12','C13','C14',
    'D1','D2','D3','D4','D5','D6','D7','D8','D9','D10',
    'D11','D12','D13','D14','D15',
    'M1','M2','M3','M4','M5','M6','M7','M8','M9',
    'V1','V2','V3','V4','V5','V6','V7','V8','V9','V10'
]

IDENTITY_COLS = [
    'TransactionID', 'DeviceType', 'DeviceInfo',
    'id_01','id_02','id_03','id_04','id_05','id_06',
    'id_07','id_08','id_09','id_10','id_11','id_12',
    'id_13','id_14','id_15','id_16','id_17','id_18',
    'id_19','id_20','id_30','id_31','id_32','id_33','id_34'
]

train_transaction = pd.read_csv(
    f"{LOCAL_PATH}/train_transaction.csv",
    usecols=TRANSACTION_COLS,
    dtype={'isFraud'       : np.int8,
           'TransactionDT' : np.int32,
           'TransactionAmt': np.float32}
)
train_identity = pd.read_csv(
    f"{LOCAL_PATH}/train_identity.csv",
    usecols=IDENTITY_COLS
)
df = train_transaction.merge(train_identity,
                              on='TransactionID',
                              how='left')
del train_transaction, train_identity
gc.collect()

print(f"    Shape   : {df.shape}")
print(f"    Memory  : {df.memory_usage().sum()/1024**2:.1f} MB")

# ── STEP 2: MISSINGNESS FLAGS ────────────────────────────────
print("\n Step 2: Creating missingness flags...")

missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100)
flag_cols = missing_pct[
    (missing_pct > 50) & (missing_pct <= 90)
].index.tolist()
drop_cols = missing_pct[missing_pct > 90].index.tolist()

# Create binary flags BEFORE imputation
flags_created = 0
for col in flag_cols:
    df[f'{col}_missing'] = df[col].isnull().astype(np.int8)
    flags_created += 1

print(f"    Created {flags_created} missingness flags")
print(f"     Will drop {len(drop_cols)} columns (>90% missing)")

# Drop high-missing columns
df.drop(columns=drop_cols, inplace=True)
gc.collect()

# ── STEP 3: AMOUNT FEATURES ──────────────────────────────────
print("\n Step 3: Transaction amount features...")

df['amt_log']     = np.log1p(
    df['TransactionAmt']).astype(np.float32)
df['amt_sqrt']    = np.sqrt(
    df['TransactionAmt']).astype(np.float32)
df['amt_decimal'] = (
    df['TransactionAmt'] -
    df['TransactionAmt'].astype(int)).astype(np.float32)
df['amt_isround'] = (
    df['amt_decimal'] == 0).astype(np.int8)

print(f"   4 amount features created")

# ── STEP 4: TIME FEATURES ────────────────────────────────────
print("\n Step 4: Time features...")

df['tx_hour']    = (
    (df['TransactionDT'] // 3600) % 24).astype(np.int8)
df['tx_day']     = (
    (df['TransactionDT'] // (3600*24)) % 7).astype(np.int8)
df['tx_week']    = (
    (df['TransactionDT'] // (3600*24*7))).astype(np.int16)
df['is_night']   = (
    (df['tx_hour'] >= 22) |
    (df['tx_hour'] <= 6)).astype(np.int8)
df['is_weekend'] = (
    df['tx_day'] >= 5).astype(np.int8)

# Fraud rate by hour (quick check)
night_fraud = df[df['is_night']==1]['isFraud'].mean()*100
day_fraud   = df[df['is_night']==0]['isFraud'].mean()*100
print(f"   ✅ 5 time features created")
print(f"    Night fraud rate : {night_fraud:.2f}%")
print(f"     Day fraud rate   : {day_fraud:.2f}%")

# ── STEP 5: CARD AGGREGATION FEATURES ───────────────────────
print("\n Step 5: Card aggregation features...")

for col in ['card1', 'card2']:
    # Fill NaN in card columns first
    df[col] = df[col].fillna(-1)
    
    grp = df.groupby(col)['TransactionAmt']
    df[f'{col}_count'] = grp.transform('count').fillna(0).astype(np.int32)
    df[f'{col}_mean']  = grp.transform('mean').fillna(0).astype(np.float32)
    df[f'{col}_std']   = grp.transform('std').fillna(0).astype(np.float32)
    df[f'{col}_ratio'] = (
        df['TransactionAmt'] /
        (df[f'{col}_mean'] + 1)).fillna(0).astype(np.float32)


gc.collect()
print(f"   8 card aggregation features created")

# ── STEP 6: EMAIL FEATURES ───────────────────────────────────
print("\n Step 6: Email features...")

high_risk = ['gmail.com','yahoo.com','hotmail.com',
             'anonymous.com','mail.com']
df['email_match']    = (
    df['P_emaildomain'] ==
    df['R_emaildomain']).astype(np.int8)
df['email_highrisk'] = (
    df['P_emaildomain'].isin(high_risk)).astype(np.int8)
df['email_missing']  = (
    df['P_emaildomain'].isnull()).astype(np.int8)

print(f"   3 email features created")

# ── STEP 7: IMPUTE MISSING VALUES ───────────────────────────
print("\n Step 7: Imputing remaining missing values...")

num_imp = cat_imp = 0
for col in df.columns:
    if df[col].isnull().sum() > 0:
        if df[col].dtype == 'object':
            df[col] = df[col].fillna('Unknown')
            cat_imp += 1
        else:
            df[col] = df[col].fillna(
                df[col].median())
            num_imp += 1

print(f"    Imputed {num_imp} numerical columns (median)")
print(f"    Imputed {cat_imp} categorical columns (Unknown)")

# ── STEP 8: ENCODE CATEGORICALS ─────────────────────────────
print("\n Step 8: Encoding categorical features...")

cat_cols = df.select_dtypes(
    include=['object']).columns.tolist()
le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(
        df[col].astype(str)).astype(np.int16)

print(f"  Encoded {len(cat_cols)} categorical columns")

gc.collect()

# ── STEP 9: FINAL SUMMARY ───────────────────────────────────
total_new = (flags_created + 4 + 5 + 8 + 3)
print(f"\n{'='*55}")
print(f" FEATURE ENGINEERING COMPLETE!")
print(f"   Final shape         : {df.shape}")
print(f"   Memory usage        : "
      f"{df.memory_usage().sum()/1024**2:.1f} MB")
print(f"   Remaining NaN       : {df.isnull().sum().sum()}")
print(f"   New features added  : {total_new}")
print(f"   Missingness flags   : {flags_created}")
print(f"   Amount features     : 4")
print(f"   Time features       : 5")
print(f"   Card features       : 8")
print(f"   Email features      : 3")
print(f"{'='*55}")

# ── STEP 10: SAVE TO S3 ─────────────────────────────────────
print("\n  Step 10: Saving to S3 checkpoint...")

output_path = '/tmp/df_features.csv'
df.to_csv(output_path, index=False)

s3 = boto3.client('s3')
s3.upload_file(
    output_path,
    BUCKET,
    'processed-data/df_features.csv'
)

print(f"    Saved locally  : {output_path}")
print(f"    Uploaded to S3 : s3://{BUCKET}/processed-data/df_features.csv")
print(f"\n Data safely checkpointed!")
print(f"   Next time — reload in seconds from S3!")
print(f"   No need to redo feature engineering!")


###  What we accomplished:
Transformed 92 raw features into **144 powerful ML features**
with zero missing values — ready for model training!

### 📊 Feature Engineering Summary:
| Step | Features Created | Technique |
|------|-----------------|-----------|
| Missingness flags | 37 | Binary NaN indicators |
| Amount features | 4 | Log, sqrt, decimal, round |
| Time features | 5 | Hour, day, week, night, weekend |
| Card aggregations | 8 | Count, mean, std, ratio |
| Email features | 3 | Match, high-risk, missing |
| **Total new** | **57** | |

### 🔑 Key Takeaways:

**1. Night transactions = higher fraud risk 🌙**
Night fraud rate (3.81%) is 15% higher than day (3.30%).
Our `is_night` binary feature captures this pattern directly.

**2. 144 clean features ready for XGBoost 🎯**
Zero NaN values remaining — no imputation needed at model time.
Memory optimized at 357MB — runs smoothly on our instance.

**3. S3 checkpoint saves all our work ☁️**
Processed data saved to S3 permanently.
Next notebook loads features in seconds — no reprocessing!

**4. Feature count: 92 → 144 (+57 features = +62%) 📈**
More high-quality features = better model performance.
We are now set up to achieve our target AUC-ROC > 0.96!

### ➡️ Next Step:
**Notebook 2: Model Training!**
We train XGBoost + LightGBM + ensemble on our 144 features,
use SHAP for explainability, and target AUC-ROC > 0.96!

## **Version 2: Enhanced Feature Engineering**

### **Goal: Push AUC-ROC from 0.9533 → 0.97+**

### 3 targeted upgrades:
| Upgrade | Strategy | Expected Gain |
|---------|----------|---------------|
| **1** | Load all V columns (V1-V339) | +0.01-0.02 AUC |
| **2** | Velocity + interaction features | +0.02-0.03 AUC |
| **3** | Save enhanced features to S3 | ready for tuning |

> 💡 Top Kaggle solutions use ALL V columns.
> We previously loaded only V1-V10 to save memory.
> Now on ml.t3.large (4GB) we load all 339 V columns!

In [3]:
# ============================================================
# CELL V2: Enhanced Feature Engineering: Smart Batch Loading
# Goal: Push AUC-ROC from 0.9533 → 0.97+
# Strategy: Load V columns in batches to avoid RAM crash
# ============================================================

import pandas as pd
import numpy as np
import gc
import boto3
from sklearn.preprocessing import LabelEncoder

print("=" * 55)
print("  ENHANCED FEATURE ENGINEERING V2")
print("=" * 55)

LOCAL_PATH = "/home/sagemaker-user/fraud-detection-platform/data/raw"
BUCKET     = "fraud-detection-mlproject-armand"

# ── STEP 1: Load core columns ────────────────────────────────
print("\n Step 1: Loading core columns...")

CORE_COLS = [
    'TransactionID', 'isFraud', 'TransactionDT',
    'TransactionAmt', 'ProductCD',
    'card1','card2','card3','card4','card5','card6',
    'addr1','addr2','dist1','dist2',
    'P_emaildomain','R_emaildomain',
    'C1','C2','C3','C4','C5','C6','C7','C8',
    'C9','C10','C11','C12','C13','C14',
    'D1','D2','D3','D4','D5','D6','D8','D9',
    'D10','D11','D15',
    'M1','M2','M3','M4','M5','M6','M7','M8','M9'
]

df = pd.read_csv(
    f"{LOCAL_PATH}/train_transaction.csv",
    usecols=CORE_COLS,
    dtype={
        'isFraud'       : np.int8,
        'TransactionDT' : np.int32,
        'TransactionAmt': np.float32
    }
)
print(f"    Core shape  : {df.shape}")
print(f"    Memory      : {df.memory_usage().sum()/1024**2:.1f} MB")

# ── STEP 2: Load V columns in 3 smart batches ────────────────
print("\n Step 2: Loading V columns in batches...")

# Batch 1: V1-V100 (most important according to Kaggle)
v_batch1 = [f'V{i}' for i in range(1, 101)]
v_batch1 = ['TransactionID'] + v_batch1

df_v1 = pd.read_csv(
    f"{LOCAL_PATH}/train_transaction.csv",
    usecols=lambda c: c in v_batch1,
    dtype={f'V{i}': np.float32 for i in range(1, 101)}
)
df = df.merge(df_v1, on='TransactionID', how='left')
del df_v1
gc.collect()
print(f"    Batch 1 (V1-V100)   loaded | Shape: {df.shape}")
print(f"    Memory: {df.memory_usage().sum()/1024**2:.1f} MB")

# Batch 2: V101-V200
v_batch2 = [f'V{i}' for i in range(101, 201)]
v_batch2 = ['TransactionID'] + v_batch2

df_v2 = pd.read_csv(
    f"{LOCAL_PATH}/train_transaction.csv",
    usecols=lambda c: c in v_batch2,
    dtype={f'V{i}': np.float32 for i in range(101, 201)}
)
df = df.merge(df_v2, on='TransactionID', how='left')
del df_v2
gc.collect()
print(f"    Batch 2 (V101-V200) loaded | Shape: {df.shape}")
print(f"    Memory: {df.memory_usage().sum()/1024**2:.1f} MB")

# Batch 3: V201-V339
v_batch3 = [f'V{i}' for i in range(201, 340)]
v_batch3 = ['TransactionID'] + v_batch3

df_v3 = pd.read_csv(
    f"{LOCAL_PATH}/train_transaction.csv",
    usecols=lambda c: c in v_batch3,
    dtype={f'V{i}': np.float32 for i in range(201, 340)}
)
df = df.merge(df_v3, on='TransactionID', how='left')
del df_v3
gc.collect()
print(f"    Batch 3 (V201-V339) loaded | Shape: {df.shape}")
print(f"    Memory: {df.memory_usage().sum()/1024**2:.1f} MB")

# ── STEP 3: Load identity columns ───────────────────────────
print("\n Step 3: Loading identity columns...")

IDENTITY_COLS = [
    'TransactionID', 'DeviceType', 'DeviceInfo',
    'id_01','id_02','id_03','id_04','id_05','id_06',
    'id_11','id_12','id_13','id_15','id_17',
    'id_19','id_20','id_31','id_33','id_34'
]

train_identity = pd.read_csv(
    f"{LOCAL_PATH}/train_identity.csv",
    usecols=IDENTITY_COLS
)
df = df.merge(train_identity, on='TransactionID', how='left')
del train_identity
gc.collect()

print(f"    Shape after identity: {df.shape}")
print(f"    Memory: {df.memory_usage().sum()/1024**2:.1f} MB")

# ── STEP 4: Drop >90% missing columns ───────────────────────
print("\n  Step 4: Dropping high-missing columns...")

missing_pct = df.isnull().sum() / len(df) * 100
drop_cols   = missing_pct[missing_pct > 90].index.tolist()
flag_cols   = missing_pct[
    (missing_pct > 50) & (missing_pct <= 90)
].index.tolist()

df.drop(columns=drop_cols, inplace=True)
gc.collect()
print(f"    Dropped {len(drop_cols)} columns")
print(f"    Remaining: {df.shape[1]} columns")

# ── STEP 5: Missingness flags ────────────────────────────────
print("\n Step 5: Missingness flags...")

flags = 0
for col in flag_cols:
    if col in df.columns:
        df[f'{col}_missing'] = df[col].isnull().astype(np.int8)
        flags += 1
print(f"    {flags} flags created")

# ── STEP 6: Amount features ──────────────────────────────────
print("\n Step 6: Amount features...")

df['amt_log']     = np.log1p(df['TransactionAmt']).astype(np.float32)
df['amt_sqrt']    = np.sqrt(df['TransactionAmt']).astype(np.float32)
df['amt_decimal'] = (
    df['TransactionAmt'] -
    df['TransactionAmt'].astype(int)
).astype(np.float32)
df['amt_isround'] = (df['amt_decimal']==0).astype(np.int8)
print(f"    4 amount features")

# ── STEP 7: Time features ────────────────────────────────────
print("\n Step 7: Time features...")

df['tx_hour']    = ((df['TransactionDT']//3600)%24).astype(np.int8)
df['tx_day']     = ((df['TransactionDT']//(3600*24))%7).astype(np.int8)
df['tx_week']    = ((df['TransactionDT']//(3600*24*7))).astype(np.int16)
df['is_night']   = (
    (df['tx_hour']>=22)|(df['tx_hour']<=6)
).astype(np.int8)
df['is_weekend'] = (df['tx_day']>=5).astype(np.int8)
print(f"    5 time features")

# ── STEP 8: Card aggregation + velocity features ─────────────
print("\n Step 8: Card aggregation + velocity features...")

for col in ['card1','card2']:
    df[col] = df[col].fillna(-1)
    grp = df.groupby(col)['TransactionAmt']
    df[f'{col}_count'] = grp.transform('count').astype(np.int32)
    df[f'{col}_mean']  = grp.transform('mean').fillna(0).astype(np.float32)
    df[f'{col}_std']   = grp.transform('std').fillna(0).astype(np.float32)
    df[f'{col}_max']   = grp.transform('max').fillna(0).astype(np.float32)
    df[f'{col}_ratio'] = (
        df['TransactionAmt'] /
        (df[f'{col}_mean']+1)
    ).fillna(0).astype(np.float32)

df['card1_amt_zscore'] = (
    (df['TransactionAmt'] - df['card1_mean']) /
    (df['card1_std'] + 1)
).fillna(0).astype(np.float32)

df['addr1'] = df['addr1'].fillna(-1)
df['addr1_count'] = df.groupby('addr1')['TransactionAmt']\
    .transform('count').fillna(0).astype(np.int32)

gc.collect()
print(f"    11 card + velocity features")

# ── STEP 9: Interaction features ────────────────────────────
print("\n Step 9: Interaction features...")

df['amt_x_hour']    = (df['amt_log'] * df['tx_hour']).astype(np.float32)
df['count_x_ratio'] = (df['card1_count'] * df['card1_ratio']).astype(np.float32)
df['amt_above_max'] = (df['TransactionAmt'] > df['card1_max']).astype(np.int8)
print(f"    3 interaction features")

# ── STEP 10: Email features ──────────────────────────────────
print("\n Step 10: Email features...")

high_risk = ['gmail.com','yahoo.com','hotmail.com',
             'anonymous.com','mail.com']
df['email_match']    = (df['P_emaildomain']==df['R_emaildomain']).astype(np.int8)
df['email_highrisk'] = df['P_emaildomain'].isin(high_risk).astype(np.int8)
df['email_missing']  = df['P_emaildomain'].isnull().astype(np.int8)
print(f"    3 email features")

# ── STEP 11: Impute + encode ─────────────────────────────────
print("\n Step 11: Imputing + encoding...")

for col in df.columns:
    if df[col].isnull().sum() > 0:
        if df[col].dtype == 'object':
            df[col] = df[col].fillna('Unknown')
        else:
            df[col] = df[col].fillna(df[col].median())

cat_cols = df.select_dtypes(include=['object']).columns.tolist()
le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(
        df[col].astype(str)
    ).astype(np.int16)

gc.collect()
print(f"    {len(cat_cols)} columns encoded")
print(f"    0 NaN remaining: {df.isnull().sum().sum()}")

# ── STEP 12: Final summary ───────────────────────────────────
v_count = len([c for c in df.columns if c.startswith('V')])
print(f"\n{'='*55}")
print(f" V2 FEATURE ENGINEERING COMPLETE!")
print(f"   Final shape    : {df.shape}")
print(f"   Memory         : {df.memory_usage().sum()/1024**2:.1f} MB")
print(f"   V columns      : {v_count} ← was 10 before!")
print(f"   Total features : {df.shape[1]}")
print(f"   NaN remaining  : {df.isnull().sum().sum()}")
print(f"{'='*55}")

# ── STEP 13: Save to S3 ──────────────────────────────────────
print("\n☁️  Saving V2 features to S3...")

output_path = '/tmp/df_features_v2.csv'
df.to_csv(output_path, index=False)

s3 = boto3.client('s3')
s3.upload_file(output_path, BUCKET,
               'processed-data/df_features_v2.csv')

print(f"    Saved: s3://{BUCKET}/processed-data/df_features_v2.csv")
print(f"\n Ready for V2 model training — target AUC > 0.97!")

  ENHANCED FEATURE ENGINEERING V2

 Step 1: Loading core columns...
    Core shape  : (590540, 51)
    Memory      : 221.3 MB

 Step 2: Loading V columns in batches...
    Batch 1 (V1-V100)   loaded | Shape: (590540, 151)
    Memory: 446.6 MB
    Batch 2 (V101-V200) loaded | Shape: (590540, 251)
    Memory: 671.9 MB
    Batch 3 (V201-V339) loaded | Shape: (590540, 390)
    Memory: 985.0 MB

 Step 3: Loading identity columns...
    Shape after identity: (590540, 408)
    Memory: 1066.1 MB

  Step 4: Dropping high-missing columns...
    Dropped 1 columns
    Remaining: 407 columns

 Step 5: Missingness flags...
    187 flags created

 Step 6: Amount features...
    4 amount features

 Step 7: Time features...
    5 time features

 Step 8: Card aggregation + velocity features...
    11 card + velocity features

 Step 9: Interaction features...
    3 interaction features

 Step 10: Email features...
    3 email features

 Step 11: Imputing + encoding...
    21 columns encoded
    0 NaN rem

###  Results Summary:
| Metric | V1 (Before) | V2 (Now) | Improvement |
|--------|-------------|----------|-------------|
| **Total features** | 144 | 621 | +477 features! |
| **V columns** | 10 | 498 | +488 V columns! |
| **Missingness flags** | 37 | 187 | +150 new signals! |
| **Memory** | 357MB | 1,140MB | Controlled ✅ |
| **NaN remaining** | 0 | 0 | Perfect ✅ |
| **RAM peak** | 406MB | 1,066MB | Under 4GB ✅ |

###  Key Takeaways:

**1. V columns: 10 → 498 — game changer!**
We went from 10 V columns to 498!
These are Vesta Corporation's proprietary fraud signals —
the most powerful features in the entire dataset.
Top Kaggle solutions scoring 0.98+ AUC use ALL V columns.
This single change is expected to push us from 0.9533 → 0.97+!

**2. Batched loading worked perfectly**
```
Step 1 Core cols  : 221MB  ← safe start
Step 2 V1-V100    : 447MB  ← controlled growth
Step 2 V101-V200  : 672MB  ← still safe
Step 2 V201-V339  : 985MB  ← under 1GB!
Step 3 Identity   : 1,066MB ← well under 4GB limit
```
Memory stayed under 2GB at all times —
the batched strategy worked exactly as planned!

**3. 187 missingness flags created**
Up from 37 flags in V1 → 187 flags in V2!
Each flag is a binary fraud signal capturing
"this field was empty for this transaction."
More V columns = more missing patterns = more signals!

**4. 621 total features ready for XGBoost**
```
V1  : 144 features → AUC 0.9533
V2  : 621 features → Expected AUC 0.97+
```
4.3x more features with 3.2x more memory —
extremely efficient use of our ml.t3.large instance!

**5. Saved to S3 — permanently safe ☁️**
```
s3://fraud-detection-mlproject-armand/
    processed-data/df_features_v2.csv
```


### ➡️ Next Step:
Open `02_model_training.ipynb` and train V2 models
on our 621 features — targeting AUC-ROC > 0.97!
```
